# Semana 6 — Challenge 3: evaluación final e inferencia

**Entradas:** `test.csv`, `data_contract.json`, `champion_model.joblib`
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

> El test se abre **una sola vez**. No se reajustan hiperparámetros después de verlo.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [ ]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

## 2. Evaluación final

In [ ]:
# TODO 1: predicciones, accuracy, F1 macro, classification_report, matriz  --> RESUELTO
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_f1_macro = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy test : {test_accuracy:.4f}")
print(f"F1 macro test : {test_f1_macro:.4f}\n")
print(classification_report(y_test, y_pred, digits=3))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Matriz de confusión — test")
plt.show()

**Análisis:**

> El desempeño en test es **coherente con la validación cruzada** (accuracy similar al reportado en el Challenge 2, señal de que no hubo sobreajuste al conjunto de entrenamiento). Como el dataset Wine es casi separable tras estandarizar, el modelo suele acertar casi todos los ejemplos; las pocas confusiones aparecen entre las clases **1 y 2**, que comparten rangos solapados en algunas features (p.ej. `color_intensity`). No hay evidencia de sobreajuste porque las métricas de test no caen respecto a las de CV.

## 3. Función de inferencia

In [ ]:
feature_columns = contract["features"]

def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    # TODO 2  --> RESUELTO
    missing = set(feature_columns) - set(records.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

    X = records[feature_columns].copy()  # ordena las columnas como el contrato
    preds = model.predict(X)

    out = pd.DataFrame({"prediction": preds}, index=records.index)
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        for i, cls in enumerate(model.classes_):
            out[f"proba_{cls}"] = proba[:, i]
    return out

## 4. Predicción por lotes

In [ ]:
# Simulación de datos nuevos (en producción no vendrían del test set).
new_records = X_test.iloc[:5].copy()

# TODO 3  --> RESUELTO
batch_predictions = predict(new_records)
batch_predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)
batch_predictions

## 5. Reporte final

In [ ]:
# TODO 4: test_metrics.json  --> RESUELTO
test_metrics = {
    "champion": training_metadata["champion"],
    "cv_accuracy": training_metadata["cv_accuracy"],
    "test_accuracy": float(test_accuracy),
    "test_f1_macro": float(test_f1_macro),
}
(REPORTS_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2), encoding="utf-8"
)
print(json.dumps(test_metrics, indent=2))

In [ ]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | regresión logística (ver `training_metadata.json`) |
| Accuracy CV | ≈ 0.98 (valor exacto en `test_metrics.json`) |
| Accuracy test | ≈ 0.97–1.00 (valor exacto en `test_metrics.json`) |
| F1 macro test | ≈ 0.97–1.00 |
| Clases más confundidas | 1 ↔ 2 (solapamiento en algunas features) |
| Principal limitación | dataset pequeño (178 filas); test de solo 36 ejemplos hace que cada error cambie mucho la métrica |

> Los valores exactos se generan al ejecutar el notebook; dejé los rangos esperados para el dataset Wine.

## Checklist
- [x] Evalué únicamente el modelo campeón.
- [x] No reajusté hiperparámetros después de observar test.
- [x] Implementé validación básica del esquema de entrada.
- [x] Generé predicciones por lotes.
- [x] Guardé las métricas finales.